In [13]:
%matplotlib inline
import cv2
import matplotlib.pyplot as plt
import numpy as np
import gtsam
import glob
import open3d as o3d
from tqdm import tqdm

In [ ]:
cam0_images_path = "./img/machine_hall/MH_03_medium/mav0/cam0/data/*.png"
cam1_images_path = "./img/machine_hall/MH_03_medium/mav0/cam1/data/*.png"

min_features = 2000
N_FRAMES = 100
START_FRAME = 1000

# cam0 intrinsics (same for each dataset, from sensor.yaml)
fx0, fy0, cx0, cy0 = 458.654, 457.296, 367.215, 248.375
K0 = np.array([[fx0, 0.0, cx0],
               [0.0, fy0, cy0],
               [0.0, 0.0, 1.0]], dtype=np.float64)
D0 = np.array([-0.28340811, 0.07395907, 0.00019359, 1.76187114e-05], dtype=np.float64)

# cam1 intrinsics (same for each dataset, from sensor.yaml)
fx1, fy1, cx1, cy1 = 457.587, 456.134, 379.999, 255.238
K1 = np.array([[fx1, 0.0, cx1],
               [0.0, fy1, cy1],
               [0.0, 0.0, 1.0]], dtype=np.float64)
D1 = np.array([-0.28368365, 0.07451284, -0.00010473, -3.55590700e-05], dtype=np.float64)

# Body-to-sensor extrinsics
T_BS_cam0 = np.array([
    [ 0.0148655429818, -0.999880929698,   0.00414029679422, -0.0216401454975 ],
    [ 0.999557249008,   0.0149672133247,  0.025715529948,   -0.064676986768  ],
    [-0.0257744366974,  0.00375618835797, 0.999660727178,    0.00981073058949],
    [ 0.0,              0.0,              0.0,               1.0             ],
], dtype=np.float64)

T_BS_cam1 = np.array([
    [ 0.0125552670891, -0.999755099723,   0.0182237714554, -0.0198435579556 ],
    [ 0.999598781151,   0.0130119051815,  0.0251588363115,  0.0453689425024 ],
    [-0.0253898008918,  0.0179005838253,  0.999517347078,   0.00786212447038],
    [ 0.0,              0.0,              0.0,              1.0             ],
], dtype=np.float64)

# Stereo extrinsic

# Convert cam1 coords to cam0
T_c0_c1 = np.linalg.inv(T_BS_cam0) @ T_BS_cam1

# Convert cam0 coords to cam1
T_c1_c0 = np.linalg.inv(T_c0_c1)

R_c0_c1, t_c0_c1 = T_c0_c1[:3, :3], T_c0_c1[:3, 3]
R_c1_c0, t_c1_c0 = T_c1_c0[:3, :3], T_c1_c0[:3, 3]

print("Stereo extrinsic cam0 <-> cam1:")
print(f"  baseline: {np.linalg.norm(t_c0_c1) * 100:.2f} cm")
print(f"  cam1 origin in cam0 frame: [{t_c0_c1[0]:+.4f}, {t_c0_c1[1]:+.4f}, {t_c0_c1[2]:+.4f}] m")
print(f"  rotation angle: {np.degrees(np.arccos((np.trace(R_c0_c1) - 1) / 2)):.3f} deg")

# Load and sort stereo frames
STRIDE = 2 # avoid cramped trajectory
cam0_files = sorted(glob.glob(cam0_images_path))[START_FRAME:START_FRAME + N_FRAMES*STRIDE:STRIDE]
cam1_files = sorted(glob.glob(cam1_images_path))[START_FRAME:START_FRAME + N_FRAMES*STRIDE:STRIDE]

assert len(cam0_files) == N_FRAMES, f"cam0: only {len(cam0_files)} frames"
assert len(cam1_files) == N_FRAMES, f"cam1: only {len(cam1_files)} frames"
print(f"\nLoaded {N_FRAMES} stereo pairs")

# --- FAST detector (shared) ---
fast = cv2.FastFeatureDetector_create(threshold=25, nonmaxSuppression=True)


: 

In [ ]:
# --- Helper: detect new features and assign IDs ---
def detect_new_features(img, existing_ids):
    global next_feature_id
    keypoints = fast.detect(img, None)
    new_pts = []
    new_ids = []
    for kp in keypoints:
        x, y = kp.pt
        new_pts.append([x, y])
        new_ids.append(next_feature_id)
        tracked_points[next_feature_id] = (x, y)
        next_feature_id += 1
    return np.array(new_pts, dtype=np.float32).reshape(-1, 1, 2), new_ids

: 

In [ ]:
# --- Helper: load and undistort grayscale image ---
def load_gray_undistorted(path, K, D):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return cv2.undistort(img, K, D)

: 

In [ ]:
#--- Helper: stereo match with LK and forward-backward check ---

def stereo_match_lk(img_a, img_b, pts_a):
    """LK con forward-backward check. Ritorna (pts_b, valid_mask)."""
    if len(pts_a) == 0:
        return np.zeros((0, 1, 2), dtype=np.float32), np.zeros(0, dtype=bool)
    pts_b_fwd, st_fwd, _ = cv2.calcOpticalFlowPyrLK(img_a, img_b, pts_a, None)
    pts_a_bwd, st_bwd, _ = cv2.calcOpticalFlowPyrLK(img_b, img_a, pts_b_fwd, None)
    err = np.linalg.norm(pts_a.reshape(-1, 2) - pts_a_bwd.reshape(-1, 2), axis=1)
    valid = (st_fwd.flatten() == 1) & (st_bwd.flatten() == 1) & (err < FB_MAX_ERR)
    return pts_b_fwd, valid

: 

In [ ]:
# --- Stereo tracking state ---
next_feature_id = 0
tracked_points = {}
tracks_per_frame = []
FB_MAX_ERR = 1.0   # px

# Frame 0: detect on cam0 and stereo match on cam1
prev_img0 = load_gray_undistorted(cam0_files[0], K0, D0)
prev_img1 = load_gray_undistorted(cam1_files[0], K1, D1)
prev_pts0, prev_ids = detect_new_features(prev_img0, set())
pts1_0, valid_stereo_0 = stereo_match_lk(prev_img0, prev_img1, prev_pts0)

tracks_per_frame.append({
    "cam0": {pid: tuple(pt.ravel()) for pid, pt in zip(prev_ids, prev_pts0)},
    "cam1": {pid: tuple(pt.ravel())
             for pid, pt, v in zip(prev_ids, pts1_0, valid_stereo_0) if v},
})

# Temporal loop: LK optical flow on cam0 + LK cam0->cam1 for each frame
for cam0_fname, cam1_fname in tqdm(list(zip(cam0_files[1:], cam1_files[1:]))):
    frame0 = load_gray_undistorted(cam0_fname, K0, D0)
    frame1 = load_gray_undistorted(cam1_fname, K1, D1)

    # LK optical flow on cam0
    next_pts0, status, _ = cv2.calcOpticalFlowPyrLK(prev_img0, frame0, prev_pts0, None)
    status = status.flatten()
    good_new = next_pts0[status == 1]
    good_ids = [pid for pid, st in zip(prev_ids, status) if st == 1]

    # Find new features on cam0 if below threshold
    if len(good_new) < min_features:
        new_pts, new_ids = detect_new_features(frame0, set(good_ids))
        good_new = np.vstack([good_new.reshape(-1, 1, 2), new_pts])
        good_ids.extend(new_ids)

    # Stereo match cam0 -> cam1 on the current frame
    pts1_k, valid_stereo = stereo_match_lk(frame0, frame1, good_new.reshape(-1, 1, 2))

    tracks_per_frame.append({
        "cam0": {pid: pt.ravel() for pid, pt in zip(good_ids, good_new)},
        "cam1": {pid: pt.ravel()
                 for pid, pt, v in zip(good_ids, pts1_k, valid_stereo) if v},
    })

    prev_img0 = frame0
    prev_img1 = frame1
    prev_pts0 = good_new.reshape(-1, 1, 2)
    prev_ids  = good_ids

# Check on the first 5 frames
n_cam0 = [len(t["cam0"]) for t in tracks_per_frame[:5]]
n_cam1 = [len(t["cam1"]) for t in tracks_per_frame[:5]]
stereo_ratio = [f"{c1/c0:.1%}" if c0 else "-"
                for c0, c1 in zip(n_cam0, n_cam1)]
print(f"cam0 feature (primi 5 frame)   : {n_cam0}")
print(f"cam1 feature stereo (primi 5)  : {n_cam1}")
print(f"stereo match rate (primi 5)    : {stereo_ratio}")


: 

In [ ]:
# 3d mapping at frame 0

# For semplicity we assume that cam0 at frame 0 is the world frame, so P0 = K0[I|0]

P0_init = K0 @ np.hstack([np.eye(3), np.zeros((3, 1))])   # world -> cam0 px (identity)
P1_init = K1 @ T_c1_c0[:3, :]                             # world -> cam1 px

stereo_ids_0 = sorted(
    set(tracks_per_frame[0]["cam0"].keys()) & set(tracks_per_frame[0]["cam1"].keys())
)
pts_c0 = np.array([tracks_per_frame[0]["cam0"][pid] for pid in stereo_ids_0],
                  dtype=np.float64).T
pts_c1 = np.array([tracks_per_frame[0]["cam1"][pid] for pid in stereo_ids_0],
                  dtype=np.float64).T

points_h = cv2.triangulatePoints(P0_init, P1_init, pts_c0, pts_c1)
points_3d = (points_h[:3] / points_h[3]).T                # (N, 3) in cam0 coords = world
points_3d_ids = np.array(stereo_ids_0, dtype=np.int64)

# Cheirality check + outlier filter (depth < 100 m)
X_cam0 = points_3d
X_cam1 = (R_c1_c0 @ X_cam0.T + t_c1_c0.reshape(3, 1)).T
good = (
    ~np.isnan(X_cam0).any(axis=1) & ~np.isinf(X_cam0).any(axis=1)
    & (X_cam0[:, 2] > 0) & (X_cam1[:, 2] > 0)
    & (X_cam0[:, 2] < 100)
)
points_3d     = points_3d[good]
points_3d_ids = points_3d_ids[good]

print(f"Stereo init @ frame 0: {len(points_3d)} / {len(stereo_ids_0)} landmark validi")
print(f"  depth median: {np.median(points_3d[:, 2]):.2f} m")
print(f"  depth p95: {np.percentile(points_3d[:, 2], 95):.2f} m")
print(f"  depth p5: {np.percentile(points_3d[:, 2], 5):.2f} m")


: 

In [ ]:
# --- VO sliding window BA: setup + run_window_ba ---
# Conversione della BA batch (ba_medium) in Visual Odometry incrementale.
# Differenza chiave: invece di un singolo factor graph con 100 pose ottimizzato
# in batch alla fine, costruiamo un graph piccolo con le ULTIME N pose e lo
# ri-ottimizziamo ad ogni nuovo frame. La continuita' tra finestre e' garantita
# da un prior fortissimo sulla posa piu' vecchia della finestra (anchor).
#
# Limitazioni note (rispetto a un VO industriale):
#   - Niente marginalizzazione vera (Schur complement): l'anchor scarta le
#     correlazioni tra posa marginalizzata e resto. La versione completa
#     userebbe gtsam.IncrementalFixedLagSmoother o ISAM2.
#   - Niente loop closure: VO open-loop, accumula drift nel tempo.

WINDOW_SIZE = 10
LM_MAX_ITER = 10
MIN_OBS_IN_WINDOW = 2   # un landmark e' "attivo" se ha >=2 obs nella finestra

# Oggetti GTSAM riusati a ogni chiamata: intrinseche, rumori, sensor pose cam1
cal0 = gtsam.Cal3_S2(fx0, fy0, 0, cx0, cy0)
cal1 = gtsam.Cal3_S2(fx1, fy1, 0, cx1, cy1)
sensor_pose_cam1 = gtsam.Pose3(T_c0_c1)   # body_P_sensor per i fattori di cam1

base_noise   = gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
cauchy_mest  = gtsam.noiseModel.mEstimator.Cauchy(1.0)
robust_noise = gtsam.noiseModel.Robust.Create(cauchy_mest, base_noise)
anchor_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-6] * 6))


def run_window_ba(k, pose_per_frame, points_3d_map, tracks_per_frame, rms_per_frame):
    """BA sulla finestra scorrevole che termina al frame k.

    - Costruisce un graph con le pose [window_start..k] e i landmark attivi.
    - Fissa la posa piu' vecchia con un prior strong (continuita' tra finestre).
    - Modifica pose_per_frame e points_3d_map in-place.
    - Appende diagnostica a rms_per_frame[k].
    """
    # 1. Finestra
    window_start = max(0, k - WINDOW_SIZE + 1)
    window = list(range(window_start, k + 1))
    if len(window) < 2:
        return

    # 2. Active landmark: visibili in finestra, gia' triangolati, >= MIN_OBS_IN_WINDOW obs
    obs_count = {}
    for f in window:
        for cam in ("cam0", "cam1"):
            for pid in tracks_per_frame[f][cam]:
                if pid in points_3d_map:
                    obs_count[pid] = obs_count.get(pid, 0) + 1
    active_ids = [pid for pid, c in obs_count.items() if c >= MIN_OBS_IN_WINDOW]
    if not active_ids:
        return

    # 3. Costruzione graph + values
    graph = gtsam.NonlinearFactorGraph()
    values = gtsam.Values()

    # 3a. Inserisci le pose (OpenCV world->cam -> GTSAM cam->world via .inverse())
    for f in window:
        R, t = pose_per_frame[f]
        cam_pose = gtsam.Pose3(gtsam.Rot3(R), np.asarray(t).reshape(3)).inverse()
        values.insert(gtsam.symbol('c', f), cam_pose)

    # 3b. Anchor: prior FORTISSIMO sulla posa piu' vecchia. Fissa il gauge e
    # garantisce continuita' col risultato della finestra precedente.
    R_anchor, t_anchor = pose_per_frame[window_start]
    anchor_pose = gtsam.Pose3(gtsam.Rot3(R_anchor),
                              np.asarray(t_anchor).reshape(3)).inverse()
    graph.add(gtsam.PriorFactorPose3(
        gtsam.symbol('c', window_start), anchor_pose, anchor_noise
    ))

    # 3c. Inserisci i landmark attivi
    for pid in active_ids:
        X = points_3d_map[pid]
        values.insert(gtsam.symbol('p', pid),
                      gtsam.Point3(float(X[0]), float(X[1]), float(X[2])))

    # 3d. Fattori di riproiezione cam0 + cam1 per ogni osservazione di ogni active id
    cam0_idx, cam1_idx = [], []
    for pid in active_ids:
        point_sym = gtsam.symbol('p', pid)
        for f in window:
            cam_sym = gtsam.symbol('c', f)
            tk = tracks_per_frame[f]
            if pid in tk["cam0"]:
                pt = tk["cam0"][pid]
                z = gtsam.Point2(float(pt[0]), float(pt[1]))
                cam0_idx.append(graph.size())
                graph.add(gtsam.GenericProjectionFactorCal3_S2(
                    z, robust_noise, cam_sym, point_sym, cal0
                ))
            if pid in tk["cam1"]:
                pt = tk["cam1"][pid]
                z = gtsam.Point2(float(pt[0]), float(pt[1]))
                cam1_idx.append(graph.size())
                graph.add(gtsam.GenericProjectionFactorCal3_S2(
                    z, robust_noise, cam_sym, point_sym, cal1, sensor_pose_cam1
                ))

    # 4. Ottimizza LM (silenzioso, max LM_MAX_ITER iterazioni)
    params = gtsam.LevenbergMarquardtParams()
    params.setMaxIterations(LM_MAX_ITER)
    params.setVerbosity("SILENT")
    params.setVerbosityLM("SILENT")
    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, values, params)
    result = optimizer.optimize()

    # 5. Riaggiorna pose e landmark globali (GTSAM cam->world -> OpenCV world->cam)
    for f in window:
        opt_pose_gtsam = result.atPose3(gtsam.symbol('c', f))
        opt_pose_opencv = opt_pose_gtsam.inverse()
        R_new = opt_pose_opencv.rotation().matrix()
        t_new = np.asarray(opt_pose_opencv.translation()).reshape(3)
        pose_per_frame[f] = (R_new, t_new)

    for pid in active_ids:
        X = result.atPoint3(gtsam.symbol('p', pid))
        points_3d_map[pid] = np.asarray(X, dtype=np.float64)

    # 6. Diagnostica RMS sulla finestra (split per camera per matchare ba_medium)
    def rms_from_indices(indices):
        if not indices:
            return 0.0
        total_sq = 0.0
        for i in indices:
            r = graph.at(i).unwhitenedError(result)
            total_sq += float(np.sum(r ** 2))
        return (total_sq / len(indices)) ** 0.5

    rms_per_frame[k] = {
        "rms":            rms_from_indices(cam0_idx + cam1_idx),
        "rms_cam0":       rms_from_indices(cam0_idx),
        "rms_cam1":       rms_from_indices(cam1_idx),
        "n_landmarks":    len(active_ids),
        "n_factors_cam0": len(cam0_idx),
        "n_factors_cam1": len(cam1_idx),
        "window_size":    len(window),
    }


print(f"VO setup pronto:")
print(f"  WINDOW_SIZE       = {WINDOW_SIZE}")
print(f"  LM_MAX_ITER       = {LM_MAX_ITER}")
print(f"  MIN_OBS_IN_WINDOW = {MIN_OBS_IN_WINDOW}")
print(f"  anchor sigma      = 1e-6 (prior strong sulla posa piu' vecchia)")


: 

In [ ]:
# --- Unified loop: due pipeline parallele (PnP-only vs PnP+sliding window VO) ---
# Per confrontare onestamente l'effetto della BA, mantengo DUE pipeline:
#
#   PIPELINE PNP-ONLY:  pose_per_frame_pnp_only, points_3d_map_pnp
#     - PnP usa points_3d_map_pnp (mai toccato da BA)
#     - re-triangulation aggiunge a points_3d_map_pnp
#     - NIENTE run_window_ba
#
#   PIPELINE VO:        pose_per_frame, points_3d_map
#     - PnP usa points_3d_map (raffinato dalla BA della finestra precedente)
#     - re-triangulation aggiunge a points_3d_map
#     - run_window_ba ad ogni frame
#
# Entrambi condividono il front-end (tracks_per_frame, dalla cella 3): stessi
# pixel di osservazione, stessi ID. La differenza e' SOLO nel back-end.

MAX_REPROJ_ERR_INIT = 2.0   # px, filtro di accettazione per nuovi landmark

def stereo_triangulate_and_filter(pts0, pts1, T_world_cam0, T_world_cam1):
    """Triangolazione stereo + filtri cheirality + depth + reprojection.
    pts0, pts1: (2, N). Ritorna (X_world (N,3), valid_mask (N,))."""
    P_cam0 = K0 @ T_world_cam0[:3, :]
    P_cam1 = K1 @ T_world_cam1[:3, :]

    pts_h = cv2.triangulatePoints(P_cam0, P_cam1, pts0, pts1)
    X_world = (pts_h[:3] / pts_h[3]).T

    X_c0 = (T_world_cam0[:3, :3] @ X_world.T + T_world_cam0[:3, 3:4]).T
    X_c1 = (T_world_cam1[:3, :3] @ X_world.T + T_world_cam1[:3, 3:4]).T
    valid = (
        ~np.isnan(X_world).any(axis=1) & ~np.isinf(X_world).any(axis=1)
        & (X_c0[:, 2] > 0) & (X_c1[:, 2] > 0)
        & (X_c0[:, 2] < 100)
    )

    X_h = np.hstack([X_world, np.ones((len(X_world), 1))])
    proj0 = (P_cam0 @ X_h.T).T
    proj1 = (P_cam1 @ X_h.T).T
    with np.errstate(divide="ignore", invalid="ignore"):
        uv0 = proj0[:, :2] / proj0[:, 2:3]
        uv1 = proj1[:, :2] / proj1[:, 2:3]
    err0 = np.linalg.norm(uv0 - pts0.T, axis=1)
    err1 = np.linalg.norm(uv1 - pts1.T, axis=1)
    valid &= (err0 < MAX_REPROJ_ERR_INIT) & (err1 < MAX_REPROJ_ERR_INIT)

    return X_world, valid


def pnp_and_triangulate(idx, pose_dict, points_map):
    """PnP + stereo re-triangulation per UN pipeline.
    Modifica pose_dict e points_map in-place. Ritorna (n_inliers, n_added)."""
    cam0_pts = tracks_per_frame[idx]["cam0"]
    cam1_pts = tracks_per_frame[idx]["cam1"]

    # --- PnP su cam0 ---
    pts3d_list, pts2d_list = [], []
    for pid, X in points_map.items():
        if pid in cam0_pts:
            pts3d_list.append(X)
            pts2d_list.append(cam0_pts[pid])
    if len(pts3d_list) < 6:
        raise ValueError(f"Frame {idx}: solo {len(pts3d_list)} match 2D-3D su cam0")

    pts3d_np = np.asarray(pts3d_list, dtype=np.float64)
    pts2d_np = np.asarray(pts2d_list, dtype=np.float64)

    ok, rvec, tvec, inliers = cv2.solvePnPRansac(
        pts3d_np, pts2d_np, K0, distCoeffs=None, flags=cv2.SOLVEPNP_ITERATIVE
    )
    if not ok:
        raise RuntimeError(f"PnP failed for frame {idx}")

    R, _ = cv2.Rodrigues(rvec)
    t = tvec.ravel()
    pose_dict[idx] = (R, t)
    n_inliers = 0 if inliers is None else len(inliers)

    # --- Re-triangulation stereo per i nuovi id con stereo match ---
    new_ids = [pid for pid in cam0_pts.keys()
               if pid in cam1_pts and pid not in points_map]
    n_added = 0
    if len(new_ids) >= 10:
        pts0 = np.asarray([cam0_pts[pid] for pid in new_ids], dtype=np.float64).T
        pts1 = np.asarray([cam1_pts[pid] for pid in new_ids], dtype=np.float64).T

        T_world_cam0 = np.eye(4); T_world_cam0[:3, :3] = R; T_world_cam0[:3, 3] = t
        T_world_cam1 = T_c1_c0 @ T_world_cam0

        X_world, valid = stereo_triangulate_and_filter(pts0, pts1,
                                                       T_world_cam0, T_world_cam1)
        for pid, X, ok_mask in zip(new_ids, X_world, valid):
            if ok_mask:
                points_map[int(pid)] = X
                n_added += 1

    return n_inliers, n_added


# --- Init: due mappe e due dict di pose, partono dallo stesso stereo init ---
points_3d_map_pnp = {int(pid): X.astype(np.float64).copy()
                     for pid, X in zip(points_3d_ids, points_3d)}
points_3d_map     = {int(pid): X.astype(np.float64).copy()
                     for pid, X in zip(points_3d_ids, points_3d)}
pose_per_frame_pnp_only = {0: (np.eye(3), np.zeros(3))}
pose_per_frame          = {0: (np.eye(3), np.zeros(3))}
rms_per_frame           = {}

for idx in tqdm(range(1, N_FRAMES)):
    # Pipeline PnP-only (mai toccata dalla BA)
    n_inl_pnp, n_add_pnp = pnp_and_triangulate(idx, pose_per_frame_pnp_only, points_3d_map_pnp)

    # Pipeline VO (PnP + re-triangulation + sliding window BA)
    n_inl_vo, n_add_vo = pnp_and_triangulate(idx, pose_per_frame, points_3d_map)
    run_window_ba(idx, pose_per_frame, points_3d_map, tracks_per_frame, rms_per_frame)

    if idx % 10 == 0 or max(n_add_pnp, n_add_vo) > 50:
        d = rms_per_frame.get(idx, {})
        rms_str = f"rms={d.get('rms', 0):.2f}px" if d else "rms=n/a"
        print(f"Frame {idx:3d}: PnP inl={n_inl_pnp:4d} +{n_add_pnp:4d}, "
              f"VO inl={n_inl_vo:4d} +{n_add_vo:4d}, "
              f"|map_pnp|={len(points_3d_map_pnp):5d}, |map_vo|={len(points_3d_map):5d}, "
              f"{rms_str}")

# --- Riepilogo finale ---
rms_values = [d["rms"] for d in rms_per_frame.values() if d.get("rms", 0) > 0]
nlm_values = [d["n_landmarks"] for d in rms_per_frame.values()]
print(f"\nVO completato:")
print(f"  pose stimate             : {len(pose_per_frame)}")
print(f"  landmark mappa PnP-only  : {len(points_3d_map_pnp)}")
print(f"  landmark mappa VO        : {len(points_3d_map)}")
print(f"  finestre BA              : {len(rms_per_frame)}")
print(f"  RMS medio                : {np.mean(rms_values):.2f} px  (min {np.min(rms_values):.2f}, max {np.max(rms_values):.2f})")
print(f"  landmark/finestra        : media {np.mean(nlm_values):.0f}  (min {np.min(nlm_values)}, max {np.max(nlm_values)})")


: 

In [ ]:
# --- PnP-only initial estimate (pre-VO): top-down view (X-Z plane)
%matplotlib inline

def opencv_pose_to_xz(R, t):
    """Da (R, t) world->cam OpenCV alla posizione della camera nel mondo (X,Z)."""
    cam_pos = -R.T @ np.asarray(t).reshape(3)
    return cam_pos

def cam1_pos_from_cam0(R0, t0):
    """Posa di cam1 nel mondo dato R0,t0 world->cam0 in OpenCV."""
    T_world_cam0 = np.eye(4); T_world_cam0[:3, :3] = R0; T_world_cam0[:3, 3] = t0
    T_world_cam1 = T_c1_c0 @ T_world_cam0
    R1 = T_world_cam1[:3, :3]; t1 = T_world_cam1[:3, 3]
    return -R1.T @ t1

# Traiettoria PnP-only (cam0 + cam1)
pnp_cam0_xz, pnp_cam1_xz = [], []
for k in sorted(pose_per_frame_pnp_only.keys()):
    R, t = pose_per_frame_pnp_only[k]
    c0 = opencv_pose_to_xz(R, t)
    c1 = cam1_pos_from_cam0(R, t)
    pnp_cam0_xz.append((c0[2], -c0[0]))
    pnp_cam1_xz.append((c1[2], -c1[0]))
pnp_cam0_xz = np.array(pnp_cam0_xz)
pnp_cam1_xz = np.array(pnp_cam1_xz)    

# Versione zoom

landmark_xz = []
z_lim=(-6.0, 2.0)
x_lim=(-2.0, 6.0)
title=f'PnP-only initial estimate (pre-VO) - zoomed view',
savepath="./img/report/ba_advanced/initial_estimate_topdown.png"

for X in points_3d_map.values():
    if X[2] < 0:
        continue
    z, mx = X[2], -X[0]
    if z_lim[0] <= z <= z_lim[1] and x_lim[0] <= mx <= x_lim[1]:
        landmark_xz.append((z, mx))

landmark_xz = np.array(landmark_xz)

plt.figure(figsize=(14, 8))

if len(landmark_xz):
    plt.scatter(landmark_xz[:, 0], landmark_xz[:, 1], c='green', s=2, alpha=0.4,
                label=f'Landmarks ({len(landmark_xz)} in viewport)')
plt.plot(pnp_cam0_xz[:, 0], pnp_cam0_xz[:, 1], 'o-',
            color='red',        markersize=4, linewidth=1.5, label='PnP-only cam0')
plt.plot(pnp_cam1_xz[:, 0], pnp_cam1_xz[:, 1], 'o-',
            color='darkorange', markersize=4, linewidth=1.5, alpha=0.8, label='PnP-only cam1')
plt.xlim(*z_lim); plt.ylim(*x_lim)
plt.gca().set_aspect('equal')
plt.xlabel('Z (world, m)'); plt.ylabel('-X (world, m)')
plt.title(title); plt.legend(loc='best'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(savepath, dpi=150, bbox_inches="tight")
plt.show()

# Versione full
z_lim=(-5.0, 10.0)
x_lim=(-10.0, 10.0)
title=f'PnP-only initial estimate - full view',
savepath="./img/report/ba_advanced/initial_estimate_topdown_full.png"

landmark_xz = np.array([(X[2], -X[0]) for X in points_3d_map.values() if X[2] > 0])

plt.figure(figsize=(14, 8))

if len(landmark_xz):
    plt.scatter(landmark_xz[:, 0], landmark_xz[:, 1], c='green', s=2, alpha=0.4,
                label=f'Landmarks ({len(landmark_xz)} in viewport)')
plt.plot(pnp_cam0_xz[:, 0], pnp_cam0_xz[:, 1], 'o-',
            color='red',        markersize=4, linewidth=1.5, label='PnP-only cam0')
plt.plot(pnp_cam1_xz[:, 0], pnp_cam1_xz[:, 1], 'o-',
            color='darkorange', markersize=4, linewidth=1.5, alpha=0.8, label='PnP-only cam1')
plt.xlim(*z_lim); plt.ylim(*x_lim)
plt.gca().set_aspect('equal')
plt.xlabel('Z (world, m)'); plt.ylabel('-X (world, m)')
plt.title(title); plt.legend(loc='best'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(savepath, dpi=150, bbox_inches="tight")
plt.show()



: 

In [ ]:
# --- VO diagnostic: RMS per frame ---

frames        = sorted(rms_per_frame.keys())
rms_all       = np.array([rms_per_frame[k]["rms"]            for k in frames])
rms_cam0_arr  = np.array([rms_per_frame[k]["rms_cam0"]       for k in frames])
rms_cam1_arr  = np.array([rms_per_frame[k]["rms_cam1"]       for k in frames])
n_landmarks   = np.array([rms_per_frame[k]["n_landmarks"]    for k in frames])
n_factors_c0  = np.array([rms_per_frame[k]["n_factors_cam0"] for k in frames])
n_factors_c1  = np.array([rms_per_frame[k]["n_factors_cam1"] for k in frames])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(frames, rms_cam0_arr, label='cam0', color='steelblue',  linewidth=1.2)
ax.plot(frames, rms_cam1_arr, label='cam1', color='darkorange', linewidth=1.2)
ax.plot(frames, rms_all,      label='all',  color='black', linewidth=1.5, linestyle='--')
ax.set_ylabel('Reprojection RMS (px)')
ax.set_title('Sliding window BA: RMS per frame (lower = better)')
ax.legend(loc='best')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(frames, n_landmarks,  label='active landmarks',  color='green',     linewidth=1.2)
ax.plot(frames, n_factors_c0, label='factors cam0',       color='steelblue', linewidth=1.0, alpha=0.7)
ax.plot(frames, n_factors_c1, label='factors cam1',       color='darkorange', linewidth=1.0, alpha=0.7)
ax.set_xlabel('Frame index')
ax.set_ylabel('count')
ax.set_title('Window content: landmark e fattori per finestra')
ax.legend(loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./img/report/ba_advanced/vo_rms_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary testuale
print(f"VO diagnostic ({len(frames)} finestre):")
print(f"  RMS totale  : mean={np.mean(rms_all):.2f}  median={np.median(rms_all):.2f}  p95={np.percentile(rms_all, 95):.2f}  max={np.max(rms_all):.2f} px")
print(f"  RMS cam0    : mean={np.mean(rms_cam0_arr):.2f}  median={np.median(rms_cam0_arr):.2f} px")
print(f"  RMS cam1    : mean={np.mean(rms_cam1_arr):.2f}  median={np.median(rms_cam1_arr):.2f} px")
print(f"  N landmark  : mean={np.mean(n_landmarks):.0f}   min={np.min(n_landmarks)}  max={np.max(n_landmarks)}")
print(f"  N fattori/finestra (cam0+cam1) : mean={np.mean(n_factors_c0 + n_factors_c1):.0f}")


: 

In [ ]:
# --- PnP-only vs VO sliding window: top-down view (X-Z plane)
%matplotlib inline

def trajectory_xz(pose_dict):
    cam0_xz, cam1_xz = [], []
    for k in sorted(pose_dict.keys()):
        R, t = pose_dict[k]
        c0 = opencv_pose_to_xz(R, t)
        c1 = cam1_pos_from_cam0(R, t)
        cam0_xz.append((c0[2], -c0[0]))
        cam1_xz.append((c1[2], -c1[0]))
    return np.array(cam0_xz), np.array(cam1_xz)

pnp_cam0_xz, pnp_cam1_xz = trajectory_xz(pose_per_frame_pnp_only)
vo_cam0_xz,  vo_cam1_xz  = trajectory_xz(pose_per_frame)



# Versione zoom
z_lim=(-6.0, 2.0)
x_lim=(-2.0, 6.0)
title=f'PnP-only vs VO sliding window BA - zoomed view'
savepath="./img/report/ba_advanced/vo_comparison_topdown.png"

landmark_xz = np.array([(X[2], -X[0]) for X in points_3d_map.values() if X[2] > 0])

plt.figure(figsize=(14, 8))
if len(landmark_xz):
    plt.scatter(landmark_xz[:, 0], landmark_xz[:, 1], c='green', s=2, alpha=0.4,
                label=f'Final landmarks ({len(landmark_xz)} in viewport)')
plt.plot(pnp_cam0_xz[:, 0], pnp_cam0_xz[:, 1], 'o-',
            color='red',        markersize=4, linewidth=1.5, label='PnP-only cam0')
plt.plot(pnp_cam1_xz[:, 0], pnp_cam1_xz[:, 1], 'o-',
            color='darkorange', markersize=4, linewidth=1.5, alpha=0.8, label='PnP-only cam1')
plt.plot(vo_cam0_xz[:, 0], vo_cam0_xz[:, 1], 'o-',
            color='blue',       markersize=4, linewidth=1.5, label='VO cam0 (sliding window BA)')
plt.plot(vo_cam1_xz[:, 0], vo_cam1_xz[:, 1], 'o-',
            color='cyan',       markersize=4, linewidth=1.5, alpha=0.85, label='VO cam1 (sliding window BA)')
plt.xlim(*z_lim); plt.ylim(*x_lim)
plt.gca().set_aspect('equal')
plt.xlabel('Z (world, m)'); plt.ylabel('-X (world, m)')
plt.title(title); plt.legend(loc='best'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(savepath, dpi=150, bbox_inches="tight")
plt.show()


# Versione full
z_lim=(-5.0, 10.0)
x_lim=(-10.0, 10.0)
title=f'PnP-only vs VO sliding window BA - full view'
savepath="./img/report/ba_advanced/vo_comparison_topdown_full.png"

landmark_xz = []
for X in points_3d_map.values():
    if X[2] < 0:
        continue
    z, mx = X[2], -X[0]
    if z_lim[0] <= z <= z_lim[1] and x_lim[0] <= mx <= x_lim[1]:
        landmark_xz.append((z, mx))
landmark_xz = np.array(landmark_xz)

plt.figure(figsize=(14, 8))
if len(landmark_xz):
    plt.scatter(landmark_xz[:, 0], landmark_xz[:, 1], c='green', s=2, alpha=0.4,
                label=f'Final landmarks ({len(landmark_xz)} in viewport)')
plt.plot(pnp_cam0_xz[:, 0], pnp_cam0_xz[:, 1], 'o-',
            color='red',        markersize=4, linewidth=1.5, label='PnP-only cam0')
plt.plot(pnp_cam1_xz[:, 0], pnp_cam1_xz[:, 1], 'o-',
            color='darkorange', markersize=4, linewidth=1.5, alpha=0.8, label='PnP-only cam1')
plt.plot(vo_cam0_xz[:, 0], vo_cam0_xz[:, 1], 'o-',
            color='blue',       markersize=4, linewidth=1.5, label='VO cam0 (sliding window BA)')
plt.plot(vo_cam1_xz[:, 0], vo_cam1_xz[:, 1], 'o-',
            color='cyan',       markersize=4, linewidth=1.5, alpha=0.85, label='VO cam1 (sliding window BA)')
plt.xlim(*z_lim); plt.ylim(*x_lim)
plt.gca().set_aspect('equal')
plt.xlabel('Z (world, m)'); plt.ylabel('-X (world, m)')
plt.title(title); plt.legend(loc='best'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(savepath, dpi=150, bbox_inches="tight")
plt.show()


: 

In [ ]:
%matplotlib widget

from mpl_toolkits.mplot3d import Axes3D

# Traiettorie cam0 e cam1 (cam1 derivata via T_c0_c1) in coordinate mondo.
cam0_xyz, cam1_xyz = [], []
for k in sorted(pose_per_frame.keys()):
    R, t = pose_per_frame[k]
    T_world_cam0 = np.eye(4); T_world_cam0[:3, :3] = R; T_world_cam0[:3, 3] = t
    T_world_cam1 = T_c1_c0 @ T_world_cam0
    cam0_xyz.append(-R.T @ t)
    cam1_xyz.append(-T_world_cam1[:3, :3].T @ T_world_cam1[:3, 3])
cam0_xyz = np.array(cam0_xyz)
cam1_xyz = np.array(cam1_xyz)

# --- Salva .ply: cloud finale post-VO + traiettorie cam0/cam1 ---
def save_landmark_ply(points_dict, path, color, max_dist=100.0):
    pts = []
    for X in points_dict.values():
        if np.linalg.norm(X) > max_dist or X[2] < 0:
            continue
        pts.append([X[0], X[1], X[2]])
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.array(pts))
    pcd.paint_uniform_color(color)
    o3d.io.write_point_cloud(path, pcd)
    return len(pts)

n_final = save_landmark_ply(points_3d_map, "./plot/advanced/ba_advanced_final.ply", [0.1, 0.7, 0.1])
print(f"Salvati {n_final} landmark in ba_advanced_final.ply (verde)")

def save_trajectory_ply(xyz, path, color):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.paint_uniform_color(color)
    o3d.io.write_point_cloud(path, pcd)

save_trajectory_ply(cam0_xyz, "./plot/advanced/ba_advanced_cam0_traj.ply", [0.0, 0.3, 1.0])
save_trajectory_ply(cam1_xyz, "./plot/advanced/ba_advanced_cam1_traj.ply", [1.0, 0.2, 0.0])
print("Salvate traiettorie cam0/cam1 come .ply (blu/rosso)")

# Matplotlib 3D widget, solo traiettorie con rungs.
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
ax.plot(cam0_xyz[:, 0], cam0_xyz[:, 2], cam0_xyz[:, 1], 'b-o', markersize=4, label='cam0')
ax.plot(cam1_xyz[:, 0], cam1_xyz[:, 2], cam1_xyz[:, 1], 'r-o', markersize=4, label='cam1')
for p0, p1 in zip(cam0_xyz, cam1_xyz):
    ax.plot([p0[0], p1[0]], [p0[2], p1[2]], [p0[1], p1[1]],
            color='gray', linewidth=0.5, alpha=0.5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Z (m)')
ax.set_zlabel('Y (m)')
ax.set_title('Stereo VO rig trajectory (rungs = 11 cm baseline)')

all_xyz = np.vstack([cam0_xyz, cam1_xyz])
rx = all_xyz[:, 0].ptp()
ry = all_xyz[:, 1].ptp()
rz = all_xyz[:, 2].ptp()
ax.set_box_aspect([rx, rz, ry])

ax.view_init(elev=25, azim=-60)
ax.legend()
plt.tight_layout()
plt.savefig("./img/report/ba_advanced/trajectory_3d.png", dpi=150, bbox_inches="tight")
plt.show()


: 